# D_ITEMS - Silver Layer

This notebook cleans and validates the MIMIC-IV `d_items` Bronze table before storing it in the Silver layer.

`D_ITEMS` is a lookup table that describes the different ICU measurements and bedside events recorded in tables such as `chartevents` and `datetimeevents`, including the item name, category, unit and parameter type.

The focus is on text consistency, missing values and keeping `itemid` as a reliable lookup key for ICU event data.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Load Bronze table
d_items = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.d_items_raw")

print(f"Rows: {d_items.count():,}")
print(f"Columns: {len(d_items.columns)}")

d_items.printSchema()

## Data Quality Check

Missing values are reviewed before cleaning so optional fields can be kept where they still provide useful information.

In [0]:
# Check missing values
total = d_items.count()

for column in d_items.columns:
    if not column.startswith("_"):
        nulls = d_items.filter(F.col(column).isNull()).count()

        if nulls > 0:
            pct = (nulls / total) * 100
            print(f"{column}: {nulls:,} nulls ({pct:.2f}%)")

### Missing Value Findings

Most missing values are in optional metadata fields rather than the main identifier.

`unitname` is only relevant for items with a measurement unit, while the normal range fields are available for a small number of items. These values will be kept as null where they are not provided rather than being filled or removed.

In [0]:
# Check simple casing/spacing inconsistencies
check_columns = ["linksto", "category", "param_type"]

for column in check_columns:
    print(f"\n{column}")

    (
        d_items
        .filter(F.col(column).isNotNull())
        .groupBy(F.lower(F.trim(F.col(column))).alias("standard_value"))
        .agg(F.collect_set(F.col(column)).alias("original_values"))
        .filter(F.size("original_values") > 1)
        .show(truncate=False)
    )

In [0]:
# Clean text fields
text_columns = [
    "label",
    "abbreviation",
    "linksto",
    "category",
    "unitname",
    "param_type"
]

d_items_silver = d_items

for column in text_columns:
    d_items_silver = d_items_silver.withColumn(
        column,
        F.when(F.trim(F.col(column)) == "", None)
         .otherwise(F.trim(F.col(column)))
    )

print(f"D_ITEMS Silver: {d_items_silver.count():,} rows")
display(d_items_silver.limit(10))

### Silver Result

`D_ITEMS` has been cleaned with light text standardisation while keeping the original item structure and optional metadata.

The lookup key remains unchanged and all meaningful null values have been preserved.

In [0]:
# Save cleaned D_ITEMS table
(
    d_items_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.d_items")
)

print("Created: workspace.silver.d_items")

In [0]:
bronze_count = spark.table("workspace.bronze.d_items_raw").count()
silver_count = spark.table("workspace.silver.d_items").count()

print(f"Bronze: {bronze_count:,}")
print(f"Silver: {silver_count:,}")
print(f"Match: {bronze_count == silver_count}")